# Diff-in-Diff Estimation
Ahora sí puta madre, ya casi vamos a terminar. 

In [1]:
from nbsetup import *
from pipeline import Datos, muestra_comun, COVARIABLES

datos = Datos()
emparejamientos, comunes = muestra_comun(datos)

emp = emparejamientos[300]
emp.pares                          # tratada, control
emp.features.incidentes_asignados.head()  # uid, timestamp, incident_level, peso

,uid,timestamp,incident_level,peso
0,T63,2016-04-22 17:31:48,PIC,1.0
1,T26,2016-04-22 20:00:05,MIN,1.0
2,T98,2016-04-22 14:57:34,PIC,1.0
3,T33,2016-04-22 14:05:15,MIN,1.0
4,T33,2016-04-22 16:31:59,MIN,1.0


## El panel

Antes de estimar nada hay que armar la tabla larga: una fila por unidad y
periodo. Cuatro decisiones de construcción, todas discutibles — si alguna no te
convence, aquí es donde se cambia.

**El día 22 hace las veces de primero de mes.** Las Fotocívicas entraron en vigor
exactamente el 22 de abril de 2019, así que los periodos van del 22 al 22 en vez
de seguir el calendario. Con meses de calendario, abril de 2019 queda mitad pre
y mitad post y hay que decidir qué hacer con ese mes partido; así el corte cae
exacto en la frontera entre `t = -1` y `t = 0`, la ventana queda simétrica
(36 periodos de cada lado) y `t` es directamente tiempo de evento, que es el eje
del event study. Es la misma construcción que usan las ventanas de `Features`.

**Los ceros se ponen.** Un mes sin incidentes es un cero, no una fila ausente.
El panel se arma sobre el producto completo unidad × periodo.

**El conteo es flotante.** Un incidente en área compartida entre dos círculos
tratados entra con 1/2 en cada uno. Sirve para MCO sobre conteos o tasas; para
Poisson habría que replantear el reparto.

**El volumen vehicular queda fuera, y con él las regresiones en tasas.** La serie
son los radares de conteo de las casetas de peaje en la periferia: miden el flujo
de entrada y salida del área metropolitana. Como denominador de los incidentes de
un círculo de 300 m no mide exposición al riesgo de ese lugar; es un índice macro
de movilidad, y una tasa construida así no se puede interpretar.

La razón por la que estaba era corregir el choque de movilidad de la pandemia
—SEMOVI reporta una caída del 45% en tránsito—, y para eso **los efectos fijos de
tiempo sirven mejor**: absorben cualquier choque que le haya pasado a la ciudad
cada mes, sin imponer que sea proporcional al tráfico de las casetas.

La variable de resultado son entonces conteos. Para que los coeficientes se lean
—que es lo que Alberto pide al hablar de reescalar las tasas— el efecto se reporta
además como porcentaje de la media del grupo de control.

Esto **no** aplica a `afluencia_nivel`, la del Metro: esa se usa como
característica del círculo, no como serie de tiempo, así que varía entre unidades
y los efectos fijos de tiempo no la tocan.


In [2]:
# los periodos: bloques de un mes contados desde el tratamiento
corte = datos.fecha_tratamiento
inicio, fin = datos.ventana

k_min = 0
while corte + pd.DateOffset(months=k_min - 1) >= inicio:
    k_min -= 1

k_max = 0
while corte + pd.DateOffset(months=k_max + 1) <= fin + pd.Timedelta(days=1):
    k_max += 1
k_max -= 1

cortes = pd.DatetimeIndex([corte + pd.DateOffset(months=k) for k in range(k_min, k_max + 2)])

periodos = pd.DataFrame({
    "t": np.arange(k_min, k_max + 1),
    "inicio": cortes[:-1],
    "fin": cortes[1:],
    "post": (np.arange(k_min, k_max + 1) >= 0).astype(int),
})

print(f"{len(periodos)} periodos, t de {periodos.t.min()} a {periodos.t.max()}")
print(f"del {periodos.inicio.iloc[0]:%Y-%m-%d} al {periodos.fin.iloc[-1] - pd.Timedelta(days=1):%Y-%m-%d}")
periodos.query("t in (-36, -1, 0, 35)")

72 periodos, t de -36 a 35
del 2016-04-22 al 2022-04-21


,t,inicio,fin,post
0,-36,2016-04-22,2016-05-22,0
35,-1,2019-03-22,2019-04-22,0
36,0,2019-04-22,2019-05-22,1
71,35,2022-03-22,2022-04-22,1


In [3]:
# las unidades: una fila por unidad emparejada, con su par y su condición
# `par` solo dice qué control le tocó a qué tratada; el clustering va por `uid`
pares = emp.pares

unidades = pd.concat([
    pd.DataFrame({"uid": pares.tratada, "par": pares.index, "tratado": 1}),
    pd.DataFrame({"uid": pares.control, "par": pares.index, "tratado": 0}),
], ignore_index=True)

print(f"{len(unidades)} unidades: {unidades.tratado.sum()} tratadas y "
      f"{(1 - unidades.tratado).sum()} controles, en {unidades.par.nunique()} pares")
unidades.head()

186 unidades: 93 tratadas y 93 controles, en 93 pares


,uid,par,tratado
0,T36,0,1
1,T45,1,1
2,T18,2,1
3,T11,3,1
4,T68,4,1


In [4]:
# el panel
NIVELES = None   # None cuenta todos; ("PIC", "FCS") deja solo los que tuvieron
                 # consecuencias personales

asignados = emp.features.incidentes_asignados
dentro = asignados[asignados.uid.isin(unidades.uid)]
if NIVELES is not None:
    dentro = dentro[dentro.incident_level.isin(NIVELES)]

# a qué periodo pertenece cada incidente
indice = np.searchsorted(cortes, dentro.timestamp, side="right") - 1
en_ventana = (indice >= 0) & (indice < len(periodos))
dentro = dentro[en_ventana].assign(t=periodos.t.values[indice[en_ventana]])

malla = pd.MultiIndex.from_product([unidades.uid, periodos.t], names=["uid", "t"])

panel = (
    dentro.groupby(["uid", "t"]).peso.sum()
    .reindex(malla, fill_value=0.0).rename("incidentes").reset_index()
    .merge(unidades, on="uid")
    .merge(periodos[["t", "inicio", "post"]], on="t")
    .sort_values(["uid", "t"]).reset_index(drop=True)
)

# verificación: el panel tiene que sumar exactamente lo que entró
descuadre = panel.incidentes.sum() - dentro.peso.sum()

print(f"filas          : {len(panel):,}  ({panel.uid.nunique()} unidades x {len(periodos)} periodos)")
print(f"incidentes     : {panel.incidentes.sum():,.1f}")
print(f"descuadre      : {descuadre:.6f}   <- tiene que ser 0")
print(f"filas en cero  : {(panel.incidentes == 0).mean():.1%}")
print("\n"*2)
panel.head()

filas          : 13,392  (186 unidades x 72 periodos)
incidentes     : 77,102.0
descuadre      : 0.000000   <- tiene que ser 0
filas en cero  : 3.5%





,uid,t,incidentes,par,tratado,inicio,post
0,C1001,-36,6.0,53,0,2016-04-22,0
1,C1001,-35,8.0,53,0,2016-05-22,0
2,C1001,-34,11.0,53,0,2016-06-22,0
3,C1001,-33,6.0,53,0,2016-07-22,0
4,C1001,-32,6.0,53,0,2016-08-22,0


In [5]:
# las covariables del propensity score, una fila por unidad emparejada.
# Sin interactuar con nada: cuáles entran y cómo se interactúan con el tiempo es
# decisión de la especificación (el punto 3a de Alberto)
covariables = emp.features.matriz.loc[unidades.uid, COVARIABLES]
covariables.head()

,road_length_m,n_vialidades,carriles_ponderados,pct_acceso_controlado,pct_doble_sentido,pct_desnivel,n_niveles,nivel_incidentes,pct_lesionados,tendencia,hubo_fcs,afluencia_nivel
uid,,,,,,,,,,,,
T36,1199.010148,2,5.002458,0.0,1.000000,0.000000,1,6.541667,0.543524,0.166667,1,3.677836e+05
T45,599.290235,1,5.000000,0.0,1.000000,0.000000,1,5.750000,0.541063,-0.291667,1,5.049007e+05
T18,3870.815443,2,3.018694,0.0,0.103133,0.020807,2,9.277778,0.541916,2.833333,1,2.580538e+05
T11,2640.142406,3,3.269653,0.0,0.227616,0.000000,1,13.111111,0.540254,1.833333,1,1.148552e+06
T68,599.349799,1,5.780825,0.0,1.000000,0.000000,1,5.722222,0.444175,-1.458333,0,0.000000e+00


# Las especificaciones

Tres columnas, en progresión. El coeficiente de interés es el mismo en las tres
—`tratado × post`— y significa lo mismo: el cambio adicional de las tratadas por
encima de lo que cambiaron sus controles.

```
(1)  incidentes ~ tratado + post + tratado:post
(2)  incidentes ~ tratado:post + C(uid) + C(t)
(3)  incidentes ~ tratado:post + [8 covariables]:post + C(uid) + C(t)
```

**La (1) es la que contesta lo que pidió Alberto.** Sin efectos fijos sobreviven
`tratado`, `post` y la interacción, así que se puede reportar la suma
`tratado + tratado:post`, que es la brecha que queda entre tratadas y controles
*después* del tratamiento. Sus cuatro coeficientes son literalmente la tabla de
cuatro medias: intercepto = control antes, `tratado` = brecha previa, `post` = lo
que cayó la ciudad, interacción = el efecto.

**La (2) y la (3) son el resultado.** Con efectos fijos de unidad, `tratado` y
`post` son colineales con las dummies y desaparecen. El intercepto queda pero es
un artefacto de qué dummy se omitió, así que no se reporta ni se interpreta:
queda un solo coeficiente que leer.

Dummies de tiempo y no un simple `post`, porque la ventana incluye la pandemia y
un escalón único no puede representarla.

## Qué covariables entran, y por qué solo 8

Ninguna de las 12 puede entrar sin interactuar: son constantes dentro de la
unidad y los efectos fijos se las comen. Por eso van multiplicadas por `post`.

Salen las cuatro que derivan de la variable de resultado —`nivel_incidentes`,
`tendencia`, `pct_lesionados`, `hubo_fcs`—. Meter el nivel previo interactuado
con post afirma que el nivel previo determina la evolución posterior, que es
exactamente lo que el supuesto de tendencias paralelas supone que *no* pasa:
sería controlar por el mecanismo que el diseño necesita ausente. Y el "antes" de
las tratadas ya es la mitad del estimador.

Alberto dijo "menos la variable dependiente pre-tratamiento". Literalmente eso
saca dos; el criterio estricto saca las cuatro. Está la pregunta abierta con él.
`COV_DID` de abajo tiene las dos listas.

Las covariables se estandarizan antes de interactuar. No cambia el ajuste
—`afluencia_nivel` llega a millones y `pct_desnivel` vive entre 0 y 1— pero evita
que la matriz quede mal condicionada. Sus coeficientes no se reportan.

## El clustering

Por **componente traslapada**, no por unidad. Dos círculos que comparten área
registran los mismos eventos, así que no son observaciones independientes: la
correlación de sus residuales es de 0.19 en mediana y llega a 0.58. Agrupar por
unidad las trataría como bloques independientes.

Las dos particiones están anidadas —cada unidad cae entera en una componente—
así que no se hacen las dos: se usa la más gruesa. A 100 m no hay traslape y la
componente *es* la unidad, así que la misma regla sirve en los cinco radios.


In [6]:
# clusters: las unidades que comparten área van al mismo grupo
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

ids_tratadas = sorted(int(u[1:]) for u in pares.tratada)
traslapes = emp.features.tratadas.pares_traslapados
traslapes = traslapes[traslapes.unidad_a.isin(ids_tratadas)
                      & traslapes.unidad_b.isin(ids_tratadas)]

posicion = {u: i for i, u in enumerate(ids_tratadas)}
adyacencia = coo_matrix(
    (np.ones(len(traslapes)),
     ([posicion[a] for a in traslapes.unidad_a],
      [posicion[b] for b in traslapes.unidad_b])),
    shape=(len(ids_tratadas),) * 2,
)
_, etiqueta = connected_components(adyacencia, directed=False)

# cada control conserva su propio cluster: no comparte incidentes con nadie
a_componente = {f"T{u}": f"G{c}" for u, c in zip(ids_tratadas, etiqueta)}
panel["cluster"] = panel.uid.map(a_componente).fillna(panel.uid)

print(f"pares de tratadas que se traslapan : {len(traslapes)}")
print(f"unidades involucradas              : {traslapes[['unidad_a', 'unidad_b']].stack().nunique()}")
print(f"clusters                           : {panel.cluster.nunique()} (de {panel.uid.nunique()} unidades)")

pares de tratadas que se traslapan : 22
unidades involucradas              : 34
clusters                           : 165 (de 186 unidades)


In [7]:
# las covariables al panel, estandarizadas e interactuadas con post
FUERA = ["nivel_incidentes", "tendencia", "pct_lesionados", "hubo_fcs"]  # las 4 del outcome
# FUERA = ["nivel_incidentes", "tendencia"]   # <- criterio literal de Alberto: 10 covariables

COV_DID = [c for c in COVARIABLES if c not in FUERA]

z = (covariables[COV_DID] - covariables[COV_DID].mean()) / covariables[COV_DID].std()
panel = panel.merge(z.rename(columns=lambda c: f"z_{c}"), left_on="uid", right_index=True)

INTERACCIONES = []
for c in COV_DID:
    panel[f"{c}_post"] = panel[f"z_{c}"] * panel.post
    INTERACCIONES.append(f"{c}_post")

panel["treat_post"] = panel.tratado * panel.post

print(f"{len(COV_DID)} covariables: {', '.join(COV_DID)}")
print(f"panel: {panel.shape[0]:,} filas x {panel.shape[1]} columnas")

8 covariables: road_length_m, n_vialidades, carriles_ponderados, pct_acceso_controlado, pct_doble_sentido, pct_desnivel, n_niveles, afluencia_nivel
panel: 13,392 filas x 25 columnas


In [8]:
# las tres especificaciones
import statsmodels.formula.api as smf

ESPECIFICACIONES = {
    "(1) sin efectos fijos": "incidentes ~ tratado + post + tratado:post",
    "(2) + EF unidad y tiempo": "incidentes ~ treat_post + C(uid) + C(t)",
    "(3) + covariables x post": ("incidentes ~ treat_post + "
                                 + " + ".join(INTERACCIONES) + " + C(uid) + C(t)"),
}


def estrellas(p):
    return "***" if p < .01 else "**" if p < .05 else "*" if p < .1 else ""


def estimar(formula, datos=None):
    datos = panel if datos is None else datos
    r = smf.ols(formula, data=datos).fit(
        cov_type="cluster", cov_kwds={"groups": datos.cluster})
    inter = "tratado:post" if "tratado:post" in r.params else "treat_post"
    control = datos[datos.tratado == 0]
    fila = {
        "coef": r.params[inter], "EE": r.bse[inter],
        "sig": estrellas(r.pvalues[inter]),
        "tratado": r.params.get("tratado", np.nan),
        "EE_tratado": r.bse.get("tratado", np.nan),
        "post": r.params.get("post", np.nan),
        "EE_post": r.bse.get("post", np.nan),
    }
    if "tratado" in r.params:   # la suma que pidió Alberto
        s = r.t_test(f"tratado + {inter} = 0")
        fila |= {"suma": float(s.effect[0]), "EE_suma": float(np.sqrt(s.sd[0, 0])),
                 "sig_suma": estrellas(float(s.pvalue))}
    else:
        fila |= {"suma": np.nan, "EE_suma": np.nan, "sig_suma": ""}
    fila |= {
        "N": int(r.nobs), "R2": r.rsquared, "clusters": datos.cluster.nunique(),
        "media_control": control[control.post == 0].incidentes.mean(),
        "sd_control": control[control.post == 0].incidentes.std(),
    }
    fila["% del control"] = 100 * fila["coef"] / fila["media_control"]
    return fila


resultados = pd.DataFrame(
    {nombre: estimar(f) for nombre, f in ESPECIFICACIONES.items()}
).T

columnas = ["tratado", "EE_tratado", "post", "EE_post", "coef", "EE", "sig", "suma", "EE_suma",
            "sig_suma", "% del control", "media_control", "sd_control",
            "N", "R2", "clusters"]
resultados[columnas].astype({"N": int, "clusters": int}).round(4)

,tratado,EE_tratado,post,EE_post,coef,EE,sig,suma,EE_suma,sig_suma,% del control,media_control,sd_control,N,R2,clusters
(1) sin efectos fijos,0.202509,0.435498,-0.652927,0.12843,-0.147551,0.196255,,0.054958,0.597493,,-2.451248,6.019415,3.859333,13392,0.009481,165
(2) + EF unidad y tiempo,NaN,NaN,NaN,NaN,-0.147551,0.198143,,NaN,NaN,,-2.451248,6.019415,3.859333,13392,0.509413,165
(3) + covariables x post,NaN,NaN,NaN,NaN,-0.106328,0.180839,,NaN,NaN,,-1.766425,6.019415,3.859333,13392,0.513404,165


# Los cinco radios y los cuatro niveles de severidad

Lo de arriba es el recorrido a 300 m con todos los incidentes. Aquí se repite en
las veinte combinaciones que necesita la tesis: cinco radios × cuatro variables
de resultado, cada una con las tres especificaciones.

Las cuatro variables, como están definidas en el capítulo 3:

| Tabla | Nivel | Qué es |
| --- | --- | --- |
| `results-main-general` | MIN + PIC + FCS | todos los incidentes |
| `results-main-min` | MIN | sin lesionados ni fallecidos, daños materiales |
| `results-main-pic` | PIC | alarma clasificada como Urgencias Médicas |
| `results-main-fcs` | FCS | se registró un cadáver en el lugar |

Las 93 unidades tratadas son las mismas en los cinco radios —esa es la muestra
común— pero **el emparejamiento se rehace en cada uno**, porque las covariables
se miden dentro del círculo y cambian con el radio. Los controles son distintos
en cada columna y eso está bien: lo que define el estimando es la muestra
tratada, que queda fija.

Los periodos y los cortes no dependen del radio, así que se reusan los de arriba.


In [9]:
# la construcción del panel, como función, para repetirla en cada radio
def construir_panel(emparejamiento, niveles=None):
    """Panel de unidad x periodo, con clusters e interacciones ya listos."""
    pares_r = emparejamiento.pares
    unidades_r = pd.concat([
        pd.DataFrame({"uid": pares_r.tratada, "par": pares_r.index, "tratado": 1}),
        pd.DataFrame({"uid": pares_r.control, "par": pares_r.index, "tratado": 0}),
    ], ignore_index=True)

    inc = emparejamiento.features.incidentes_asignados
    inc = inc[inc.uid.isin(unidades_r.uid)]
    if niveles is not None:
        inc = inc[inc.incident_level.isin(niveles)]

    i = np.searchsorted(cortes, inc.timestamp, side="right") - 1
    dentro_ventana = (i >= 0) & (i < len(periodos))
    inc = inc[dentro_ventana].assign(t=periodos.t.values[i[dentro_ventana]])

    malla_r = pd.MultiIndex.from_product([unidades_r.uid, periodos.t], names=["uid", "t"])
    p = (inc.groupby(["uid", "t"]).peso.sum()
         .reindex(malla_r, fill_value=0.0).rename("incidentes").reset_index()
         .merge(unidades_r, on="uid")
         .merge(periodos[["t", "post"]], on="t"))

    assert abs(p.incidentes.sum() - inc.peso.sum()) < 1e-6, "el panel no cuadra"

    # clusters: las tratadas que comparten área van juntas
    ids = sorted(int(u[1:]) for u in pares_r.tratada)
    ov = emparejamiento.features.tratadas.pares_traslapados
    ov = ov[ov.unidad_a.isin(ids) & ov.unidad_b.isin(ids)]
    pos = {u: k for k, u in enumerate(ids)}
    ady = coo_matrix((np.ones(len(ov)),
                      ([pos[a] for a in ov.unidad_a], [pos[b] for b in ov.unidad_b])),
                     shape=(len(ids),) * 2)
    _, comp = connected_components(ady, directed=False)
    p["cluster"] = p.uid.map({f"T{u}": f"G{c}" for u, c in zip(ids, comp)}).fillna(p.uid)

    # covariables estandarizadas, interactuadas con post
    cov = emparejamiento.features.matriz.loc[unidades_r.uid, COV_DID]
    zc = (cov - cov.mean()) / cov.std()
    p = p.merge(zc.rename(columns=lambda c: f"z_{c}"), left_on="uid", right_index=True)
    for c in COV_DID:
        p[f"{c}_post"] = p[f"z_{c}"] * p.post

    p["treat_post"] = p.tratado * p.post
    return p


# comprobación: a 300 m con todos los incidentes tiene que dar lo mismo que arriba
prueba = construir_panel(emparejamientos[300])
print(f"filas {len(prueba):,} | incidentes {prueba.incidentes.sum():,.0f} | "
      f"clusters {prueba.cluster.nunique()}")
print(f"coincide con el panel de arriba: {abs(prueba.incidentes.sum() - panel.incidentes.sum()) < 1e-6}")

filas 13,392 | incidentes 77,102 | clusters 165
coincide con el panel de arriba: True


In [10]:
pd.options.display.max_rows = 100

In [11]:
# las 60 regresiones: 5 radios x 4 niveles x 3 especificaciones
NIVELES_TESIS = {"general": None, "MIN": ("MIN",), "PIC": ("PIC",), "FCS": ("FCS",)}

filas = []
for radio, e in emparejamientos.items():
    for etiqueta, niveles in NIVELES_TESIS.items():
        p = construir_panel(e, niveles)
        for nombre, formula in ESPECIFICACIONES.items():
            filas.append({"radio": radio, "nivel": etiqueta, "especificacion": nombre}
                         | estimar(formula, p))

resultados_radios = pd.DataFrame(filas)
print(f"{len(resultados_radios)} estimaciones")
resultados_radios

60 estimaciones


,radio,nivel,especificacion,coef,EE,sig,tratado,EE_tratado,post,EE_post,suma,EE_suma,sig_suma,N,R2,clusters,media_control,sd_control,% del control
0,100,general,(1) sin efectos fijos,0.094385,0.108900,,6.332139e-02,0.286788,-4.429510e-01,0.073251,0.157706,0.476543,,13392,0.007602,186,2.222820,2.516499,4.246170
1,100,general,(2) + EF unidad y tiempo,0.094385,0.109948,,NaN,NaN,NaN,NaN,NaN,NaN,,13392,0.550351,186,2.222820,2.516499,4.246170
2,100,general,(3) + covariables x post,0.124564,0.097052,,NaN,NaN,NaN,NaN,NaN,NaN,,13392,0.554552,186,2.222820,2.516499,5.603882
3,100,MIN,(1) sin efectos fijos,0.020012,0.085828,,4.480287e-03,0.182455,-4.426523e-01,0.061895,0.024492,0.344634,,13392,0.018817,186,1.319295,1.765086,1.516867
4,100,MIN,(2) + EF unidad y tiempo,0.020012,0.086654,,NaN,NaN,NaN,NaN,NaN,NaN,,13392,0.444272,186,1.319295,1.765086,1.516867
5,100,MIN,(3) + covariables x post,0.040894,0.074664,,NaN,NaN,NaN,NaN,NaN,NaN,,13392,0.452059,186,1.319295,1.765086,3.099682
6,100,PIC,(1) sin efectos fijos,0.075866,0.045157,*,5.794504e-02,0.113556,2.986858e-04,0.024232,0.133811,0.343242,,13392,0.001864,186,0.889486,1.201422,8.529214
7,100,PIC,(2) + EF unidad y tiempo,0.075866,0.045591,*,NaN,NaN,NaN,NaN,NaN,NaN,,13392,0.384296,186,0.889486,1.201422,8.529214
8,100,PIC,(3) + covariables x post,0.084752,0.041640,**,NaN,NaN,NaN,NaN,NaN,NaN,,13392,0.385882,186,0.889486,1.201422,9.528229
9,100,FCS,(1) sin efectos fijos,-0.001493,0.004547,,8.960573e-04,0.003624,-5.973716e-04,0.002809,-0.000597,0.061586,,13392,0.000043,186,0.014038,0.120178,-10.638298


In [12]:
# el coeficiente del DiD en las veinte combinaciones, especificación (3)
principal = resultados_radios.query("especificacion == '(3) + covariables x post'")

for columna, titulo in [("coef", "COEFICIENTE (incidentes por círculo al mes)"),
                        ("% del control", "% DE LA MEDIA DEL GRUPO DE CONTROL"),
                        ("EE", "ERROR ESTÁNDAR")]:
    print(titulo)
    print(principal.pivot(index="nivel", columns="radio", values=columna)
          .reindex(list(NIVELES_TESIS)).round(3).to_string())
    print()

COEFICIENTE (incidentes por círculo al mes)
radio      100    150    200    250    300
nivel                                     
general  0.125 -0.040 -0.091 -0.131 -0.106
MIN      0.041 -0.069 -0.068 -0.052 -0.045
PIC      0.085  0.028 -0.020 -0.085 -0.065
FCS     -0.001  0.001 -0.003  0.006  0.004

% DE LA MEDIA DEL GRUPO DE CONTROL
radio      100    150     200     250     300
nivel                                        
general  5.604 -1.371  -2.358  -2.607  -1.766
MIN      3.100 -4.023  -2.912  -1.733  -1.225
PIC      9.528  2.273  -1.326  -4.301  -2.840
FCS     -7.708  3.419 -17.403  18.608  11.617

ERROR ESTÁNDAR
radio      100    150    200    250    300
nivel                                     
general  0.097  0.109  0.131  0.158  0.181
MIN      0.075  0.082  0.102  0.118  0.140
PIC      0.042  0.055  0.067  0.077  0.080
FCS      0.004  0.005  0.005  0.006  0.006



In [13]:
# la tabla completa de una combinación, para revisar que todo esté
(resultados_radios
 .query("nivel == 'general'")
 .set_index(["radio", "especificacion"])
 [["tratado", "EE_tratado", "post", "EE_post", "coef", "EE", "sig",
   "suma", "EE_suma", "sig_suma", "% del control",
   "media_control", "sd_control", "N", "R2", "clusters"]]
 .round(4))

tratado  EE_tratado    post  EE_post    coef  \
radio especificacion                                                           
100   (1) sin efectos fijos      0.0633      0.2868 -0.4430   0.0733  0.0944   
      (2) + EF unidad y tiempo      NaN         NaN     NaN      NaN  0.0944   
      (3) + covariables x post      NaN         NaN     NaN      NaN  0.1246   
150   (1) sin efectos fijos      0.0352      0.2947 -0.3492   0.0747 -0.0639   
      (2) + EF unidad y tiempo      NaN         NaN     NaN      NaN -0.0639   
      (3) + covariables x post      NaN         NaN     NaN      NaN -0.0404   
200   (1) sin efectos fijos      0.0275      0.3260 -0.4277   0.0873 -0.0827   
      (2) + EF unidad y tiempo      NaN         NaN     NaN      NaN -0.0827   
      (3) + covariables x post      NaN         NaN     NaN      NaN -0.0911   
250   (1) sin efectos fijos      0.0460      0.3739 -0.5128   0.1200 -0.1293   
      (2) + EF unidad y tiempo      NaN         NaN     NaN      NaN -0.1293   
      (3) + covariables x post      NaN         NaN     NaN      NaN -0.1311   
300   (1) sin efectos fijos      0.2025      0.4355 -0.6529   0.1284 -0.1476   
      (2) + EF unidad y tiempo      NaN         NaN     NaN      NaN -0.1476   
      (3) + covariables x post      NaN         NaN     NaN      NaN -0.1063   

                                    EE sig    suma  EE_suma sig_suma  \
radio especificacion                                                   
100   (1) sin efectos fijos     0.1089      0.1577   0.4765            
      (2) + EF unidad y tiempo  0.1099         NaN      NaN            
      (3) + covariables x post  0.0971         NaN      NaN            
150   (1) sin efectos fijos     0.1188     -0.0287   0.5014            
      (2) + EF unidad y tiempo  0.1200         NaN      NaN            
      (3) + covariables x post  0.1090         NaN      NaN            
200   (1) sin efectos fijos     0.1385     -0.0553   0.5262            
      (2) + EF unidad y tiempo  0.1399         NaN      NaN            
      (3) + covariables x post  0.1314         NaN      NaN            
250   (1) sin efectos fijos     0.1756     -0.0833   0.5583            
      (2) + EF unidad y tiempo  0.1773         NaN      NaN            
      (3) + covariables x post  0.1583         NaN      NaN            
300   (1) sin efectos fijos     0.1963      0.0550   0.5975            
      (2) + EF unidad y tiempo  0.1981         NaN      NaN            
      (3) + covariables x post  0.1808         NaN      NaN            

                                % del control  media_control  sd_control  \
radio especificacion                                                       
100   (1) sin efectos fijos            4.2462         2.2228      2.5165   
      (2) + EF unidad y tiempo         4.2462         2.2228      2.5165   
      (3) + covariables x post         5.6039         2.2228      2.5165   
150   (1) sin efectos fijos           -2.1677         2.9486      2.7037   
      (2) + EF unidad y tiempo        -2.1677         2.9486      2.7037   
      (3) + covariables x post        -1.3713         2.9486      2.7037   
200   (1) sin efectos fijos           -2.1403         3.8656      3.0198   
      (2) + EF unidad y tiempo        -2.1403         3.8656      3.0198   
      (3) + covariables x post        -2.3579         3.8656      3.0198   
250   (1) sin efectos fijos           -2.5725         5.0275      3.4259   
      (2) + EF unidad y tiempo        -2.5725         5.0275      3.4259   
      (3) + covariables x post        -2.6074         5.0275      3.4259   
300   (1) sin efectos fijos           -2.4512         6.0194      3.8593   
      (2) + EF unidad y tiempo        -2.4512         6.0194      3.8593   
      (3) + covariables x post        -1.7664         6.0194      3.8593   

                                    N      R2  clusters  
radio especificacion                                     
100   (1) sin efectos fijos     13392

In [14]:
# significancia: dónde, si en algún lado, el efecto se distingue de cero
principal = principal.assign(t=lambda d: d.coef / d.EE)

print("ESTADÍSTICO t")
print(principal.pivot(index="nivel", columns="radio", values="t")
      .reindex(list(NIVELES_TESIS)).round(2).to_string())
print()
print("SIGNIFICANCIA  (* 10%, ** 5%, *** 1%)")
print(principal.pivot(index="nivel", columns="radio", values="sig")
      .reindex(list(NIVELES_TESIS)).replace("", "·").to_string())
print()

con_estrellas = resultados_radios[resultados_radios.sig != ""]
print(f"de las {len(resultados_radios)} estimaciones, {len(con_estrellas)} alcanzan significancia:")
if len(con_estrellas):
    print(con_estrellas[["radio", "nivel", "especificacion", "coef", "EE", "sig",
                         "% del control"]].round(4).to_string(index=False))

ESTADÍSTICO t
radio     100   150   200   250   300
nivel                                
general  1.28 -0.37 -0.69 -0.83 -0.59
MIN      0.55 -0.84 -0.66 -0.44 -0.32
PIC      2.04  0.50 -0.30 -1.09 -0.81
FCS     -0.25  0.13 -0.65  0.99  0.70

SIGNIFICANCIA  (* 10%, ** 5%, *** 1%)
radio   100 150 200 250 300
nivel                      
general   ·   ·   ·   ·   ·
MIN       ·   ·   ·   ·   ·
PIC      **   ·   ·   ·   ·
FCS       ·   ·   ·   ·   ·

de las 60 estimaciones, 3 alcanzan significancia:
 radio nivel           especificacion   coef     EE sig  % del control
   100   PIC    (1) sin efectos fijos 0.0759 0.0452   *         8.5292
   100   PIC (2) + EF unidad y tiempo 0.0759 0.0456   *         8.5292
   100   PIC (3) + covariables x post 0.0848 0.0416  **         9.5282


# Verificación contra el pipeline

Todo lo de arriba vive ahora en `scripts/pipeline/estimacion.py`, en la clase
`Panel` y la función `tabla_resultados`. El notebook conserva el raciocinio y las
pruebas; el pipeline ejecuta la versión final.

Esta sección comprueba que las dos den exactamente lo mismo. Si algún día dejan
de coincidir, es que una de las dos cambió sin la otra.


In [15]:
from pipeline import Panel, tabla_resultados, NIVELES_TESIS as NIVELES_PIPELINE

# el panel a 300 m con todos los incidentes
p = Panel(emparejamientos[300])
p.resumen()

iguales = (
    len(p.tabla) == len(panel)
    and abs(p.tabla.incidentes.sum() - panel.incidentes.sum()) < 1e-9
    and p.tabla.cluster.nunique() == panel.cluster.nunique()
    and p.covariables == COV_DID
)
print()
print(f"coincide con el panel del notebook: {iguales}")


radio 300 m | incidentes: todos
periodos: t de -36 a 35, del 2016-04-22 al 2022-04-21

  unidades      :      186  (93 pares)
  filas         :   13,392
  incidentes    :   77,102.0
  filas en cero :      3.5%
  clusters      :      165
  covariables   :        8

coincide con el panel del notebook: True


In [16]:
# las 60 estimaciones, desde el pipeline
del_pipeline = tabla_resultados(emparejamientos)

COMPARAR = ["coef", "EE", "tratado", "EE_tratado", "post", "EE_post",
            "suma", "EE_suma", "media_control", "sd_control", "R2", "N", "clusters"]
llaves = ["radio", "nivel", "especificacion"]

a = resultados_radios.set_index(llaves)[COMPARAR].astype(float).sort_index()
b = del_pipeline.set_index(llaves)[COMPARAR].astype(float).sort_index()

diferencia = (a - b).abs().max().max()
print(f"filas: notebook {len(a)}, pipeline {len(b)}")
print(f"diferencia máxima en {len(COMPARAR)} columnas: {diferencia:.2e}")
assert a.index.equals(b.index) and diferencia < 1e-9, "el pipeline no reproduce el notebook"
print("\nel pipeline reproduce el notebook exactamente")


filas: notebook 60, pipeline 60
diferencia máxima en 13 columnas: 1.77e-13

el pipeline reproduce el notebook exactamente


In [17]:
# la tabla que va a la tesis, ya desde el pipeline
principal_pipeline = del_pipeline.query("especificacion == '(3) + covariables x post'")

print("EFECTO COMO % DE LA MEDIA DEL GRUPO DE CONTROL")
print(principal_pipeline.pivot(index="nivel", columns="radio", values="% del control")
      .reindex(list(NIVELES_PIPELINE)).round(2).to_string())
print()
print(f"significativas: {(del_pipeline.sig != '').sum()} de {len(del_pipeline)}")
print(del_pipeline[del_pipeline.sig != ""]
      [["radio", "nivel", "especificacion", "coef", "EE", "sig", "% del control"]]
      .round(4).to_string(index=False))


EFECTO COMO % DE LA MEDIA DEL GRUPO DE CONTROL
radio     100   150    200    250    300
nivel                                   
general  5.60 -1.37  -2.36  -2.61  -1.77
MIN      3.10 -4.02  -2.91  -1.73  -1.22
PIC      9.53  2.27  -1.33  -4.30  -2.84
FCS     -7.71  3.42 -17.40  18.61  11.62

significativas: 3 de 60
 radio nivel           especificacion   coef     EE sig  % del control
   100   PIC    (1) sin efectos fijos 0.0759 0.0452   *         8.5292
   100   PIC (2) + EF unidad y tiempo 0.0759 0.0456   *         8.5292
   100   PIC (3) + covariables x post 0.0848 0.0416  **         9.5282
